In [179]:
# CÉLULA 1 — Setup & Configuração global

import os, sys, re, json, math, random, shutil, unicodedata, pathlib, textwrap, warnings
from datetime import datetime
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ===== Reprodutibilidade =====
SEED = 42
random.seed(SEED); np.random.seed(SEED)

def hr(title=None, ch="="):
    print("\n" + ch*100)
    if title: print(title, "\n" + ch*100)

def resolve_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "code").exists() and (
            (p / "requirements.txt").exists() or (p / "README.md").exists() or (p / ".git").exists()
        ):
            return p
    return start

CWD = Path.cwd()
ROOT = resolve_repo_root(CWD)
print("ROOT:", ROOT)

# Caminhos padrão
DATASET_CANDIDATES = [
    ROOT / "code" / "notebooks" / "dataset_unificado.csv",
    ROOT / "code" / "notebooks" / "dataset" / "dataset_unificado.csv",
    Path("/mnt/data/dataset_unificado.csv"),
]
FLUXOS_CANDIDATES = [
    ROOT / "code" / "resources" / "fluxos.yaml",
    Path("/mnt/data/fluxos.yaml"),
]

def pick_first_exists(paths):
    for p in paths:
        if p.exists():
            return p
    return None

DATASET_PATH = pick_first_exists(DATASET_CANDIDATES)
FLUXOS_PATH  = pick_first_exists(FLUXOS_CANDIDATES)

OUT_DIR = ROOT / "code" / "notebooks" / "outputs" / "augmentation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --------- FLAGS ----------
ENABLE_BACKTRANSLATION = False
ENABLE_ACTIVE_LEARNING = False
PROMOTE_CANONICAL_AS_OFFICIAL = False
TEST_RATIO = 0.12
DEV_RATIO  = 0.08
MAX_SYNTH_PER_INTENT_STEP = 2000
OOV_MAX_RATE = 0.06

print("DATASET_PATH:", DATASET_PATH)
print("FLUXOS_PATH:", FLUXOS_PATH)
print("OUT_DIR:", OUT_DIR)
print("Flags => backtranslation:", ENABLE_BACKTRANSLATION, "| active_learning:", ENABLE_ACTIVE_LEARNING, "| promote:", PROMOTE_CANONICAL_AS_OFFICIAL)


ROOT: c:\Users\win\Documents\GitHub\2025-2A-T07-CC11-G04
DATASET_PATH: c:\Users\win\Documents\GitHub\2025-2A-T07-CC11-G04\code\notebooks\dataset\dataset_unificado.csv
FLUXOS_PATH: c:\Users\win\Documents\GitHub\2025-2A-T07-CC11-G04\code\resources\fluxos.yaml
OUT_DIR: c:\Users\win\Documents\GitHub\2025-2A-T07-CC11-G04\code\notebooks\outputs\augmentation
Flags => backtranslation: False | active_learning: False | promote: False


In [180]:
# CÉLULA 2 — Carregar insumos (dataset + fluxos.yaml) com autodetecção de colunas

import yaml, re

if DATASET_PATH is None or not DATASET_PATH.exists():
    raise FileNotFoundError(
        "dataset_unificado.csv não encontrado. Ajuste DATASET_CANDIDATES na Célula 1."
    )

if FLUXOS_PATH is None or not FLUXOS_PATH.exists():
    # não é fatal; tocamos sem fluxos
    fluxos = {}
else:
    fluxos = yaml.safe_load(FLUXOS_PATH.read_text(encoding="utf-8")) or {}

df0 = pd.read_csv(DATASET_PATH)
orig_cols = df0.columns.tolist()
df0.columns = [c.strip().lower() for c in df0.columns]

# ---- Autodetecta coluna de TEXTO
TEXT_CANDIDATES = [
    "mensagem_clean","mensagem","texto","message","text","conteudo","conteúdo","frase","input",
    "utterance","sentence","pergunta","fala","user_text","customer_message","msg","body"
]
TEXT_COL = next((c for c in TEXT_CANDIDATES if c in df0.columns), None)
if TEXT_COL is None:
    raise ValueError(
        "Não encontrei a coluna de texto. Colunas existentes: "
        f"{orig_cols}\n"
        "Renomeie sua coluna de mensagens para 'mensagem_clean' ou 'mensagem'."
    )

# ---- Coluna de INTENÇÃO (pode não existir)
INTENT_CANDIDATES = [
    "intent","intencao","intenção","classe","class","label","rotulo","rótulo","categoria",
    "tag","tag_intencao","intent_name","gold_intent","target","y","classe_intencao"
]
INT_COL = next((c for c in INTENT_CANDIDATES if c in df0.columns), None)

# YAML → intents conhecidas
yaml_intents = set()
def walk_yaml(d):
    if isinstance(d, dict):
        for k,v in d.items():
            if k.lower() in ("intent","intencao","intenção") and isinstance(v, str):
                yaml_intents.add(v)
            walk_yaml(v)
    elif isinstance(d, list):
        for x in d: walk_yaml(x)
walk_yaml(fluxos)
yaml_intents = {str(x).strip().lower() for x in yaml_intents if str(x).strip()}

# Normaliza nomes finais
REN = {}
if TEXT_COL != "texto": REN[TEXT_COL] = "texto"
if INT_COL and INT_COL != "intent": REN[INT_COL] = "intent"
if REN: df0 = df0.rename(columns=REN)

# Clean básico
def clean_text(s: str) -> str:
    if not isinstance(s, str): return ""
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s

df0["texto"] = df0["texto"].astype(str).map(clean_text)

NEEDS_AUTO_LABEL = "intent" not in df0.columns
print("Colunas lidas:", orig_cols)
print("TEXTO em:", TEXT_COL, "→ usando 'texto'")
print("INTENT:", "ausente (vamos auto-rotular | Célula 2B)" if NEEDS_AUTO_LABEL else "encontrada → 'intent'")


Colunas lidas: ['id', 'ts', 'texto', 'intent', 'origem', 'arquivo_origem', 'is_ruido', 'fonte', 'split', 'is_synth_test']
TEXTO em: texto → usando 'texto'
INTENT: encontrada → 'intent'


In [181]:
# CÉLULA 2B — Auto-rotulagem heurística (se 'intent' não existe)

import unicodedata
def strip_accents(s):
    return "".join(ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch))

def norm(s: str) -> str:
    s = strip_accents((s or "").lower())
    s = re.sub(r"\s+", " ", s).strip()
    return s

CEP_RX = re.compile(r"\b\d{5}-?\d{3}\b")
CATEG_KEYS = ["vestido","saia","blazer","calca","calça","body","camisa","jaqueta","short","shorts","midi","alfaiataria","linho","viscose"]
PAG_KEYS   = ["preco","preço","valor","quanto","custa","pix","boleto","parcel","parcela","cartao","cartão","juros","3x","2x","4x"]
TAM_KEYS   = ["tamanho","tam","pp","p","m","g","gg","36","38","40","42","44","46","serve","veste","busto","quadril","cintura","tabela de medidas","modelagem"]
SUG_KEYS   = ["look","combina","sugere","indica","sugestao","sugestão","estilo","ocasiao","ocasião"]
FRETE_KEYS = ["frete","prazo","entrega","sedex","pac","correios","transportadora","rastreamento","rastreio","chega","chegada"]
TROCA_KEYS = ["troca","devolucao","devolução","defeito","garantia","nao serviu","não serviu"]
PLAT_KEYS  = ["checkout","pagamento recusado","carrinho","bug","erro","site","plataforma","trava","travando"]
HUMAN_KEYS = ["atendente","humano","pessoa","telefone","whatsapp","numero","número","falar com"]
B2B_KEYS   = ["cnpj","atacado","lojista","nota fiscal","pedido minimo","pedido mínimo","b2b"]
POSV_KEYS  = ["chegou errado","faltando","troca por defeito","descosturou","rasgou"]
GRAT_KEYS  = ["obg","obrig","obrigada","obrigado","valeu","agradeco","agradeço","muito obrigada","muito obrigado"]
SAUD_KEYS  = ["oi","oie","ola","olá","bom dia","boa tarde","boa noite","e ai","e aí"]
DESP_KEYS  = ["tchau","ate","até","vlw","valeu","obg","brigada","brigadinha","ate mais","até mais"]

def intent_by_rules(t: str) -> str:
    x = norm(t)
    if len(x) <= 1: return "outros"
    if any(k in x for k in SAUD_KEYS): return "saudacao"
    if any(k in x for k in DESP_KEYS): return "despedida"
    if any(k in x for k in GRAT_KEYS): return "agradecimento"
    if CEP_RX.search(x) and any(k in x for k in FRETE_KEYS): return "prazo_entrega"
    if "frete" in x and CEP_RX.search(x): return "frete_prazo"
    if any(k in x for k in ["frete","prazo"]) and ("entrega" in x or "chega" in x): return "prazo_entrega"
    if any(k in x for k in ["modelagem","tabela de medidas","veste grande","veste pequeno"]): return "tamanho_modelagem"
    if any(k in x for k in TAM_KEYS) and any(k in x for k in CATEG_KEYS): return "duvida_tamanho"
    if any(k in x for k in TAM_KEYS) and ("produto" in x or "serve" in x or "veste" in x): return "duvida_tamanho"
    if any(k in x for k in SUG_KEYS):
        return "pedido_sugestao" if ("look" in x or "sugest" in x or "indica" in x) else "styling_sugestao_look"
    if any(k in x for k in ["procuro","tem","quero"]) and any(k in x for k in CATEG_KEYS): return "buscar_produto_por_categoria"
    if any(k in x for k in ["tem","procuro"]) and (("preto" in x) or any(k in x for k in TAM_KEYS)): return "buscar_produto_por_nome"
    if "ainda tem" in x or ("tem" in x and any(k in x for k in TAM_KEYS)): return "disponibilidade"
    if any(k in x for k in PAG_KEYS):
        if any(k in x for k in ["pix","boleto","parcel","cartao","cartão"]): return "formas_pagamento"
        return "preco_pagamento"
    if any(k in x for k in ["pedido","rastreamento","rastreio","codigo","código"]) and any(k in x for k in ["cadê","nao atualiza","não atualiza","status"]): return "status_pedido"
    if any(k in x for k in TROCA_KEYS): return "troca_devolucao"
    if any(k in x for k in ["linho","viscose","algodao","algodão","encolhe","lavar","passar","cuidado"]): return "materiais_cuidados"
    if any(k in x for k in ["link","site","loja fisica","loja física","endereco","endereço","como compro","onde compro"]): return "onde_comprar"
    if any(k in x for k in PLAT_KEYS): return "erros_plataforma"
    if any(k in x for k in HUMAN_KEYS): return "falar_com_humano"
    if "primeira" in x and any(k in x for k in ["compra","pedido"]) and any(k in x for k in ["cupom","desconto"]): return "cupom_primeira"
    if any(k in x for k in B2B_KEYS): return "cliente_loja_b2b"
    if any(k in x for k in POSV_KEYS): return "pos_venda"
    return "outros"

if NEEDS_AUTO_LABEL:
    df0["intent"] = df0["texto"].map(intent_by_rules)
    df0["intent_source"] = "heuristic"
else:
    df0["intent_source"] = "gold"

print("Auto-rotulagem:", "FEITA (heurística)" if NEEDS_AUTO_LABEL else "N/A (já havia 'intent')")
print(df0["intent"].value_counts().head(10))


Auto-rotulagem: N/A (já havia 'intent')
intent
outros                          279
saudacao                        180
buscar_produto_por_categoria    151
despedida                       118
duvida_tamanho                   88
disponibilidade                  83
buscar_produto_por_nome          39
onde_comprar                     28
pedido_sugestao                  26
troca_devolucao                  17
Name: count, dtype: int64


In [182]:
# CÉLULA 3 — StyleProfile (vocabulário)

EMOJI_PATTERN = re.compile(r"[\U00010000-\U0010ffff]", flags=re.UNICODE)

ABBREV_WHITELIST = {
    "vc","vcs","pq","qdo","qnt","qto","obg","obrig","blz","td","tmb","ctz","hj","amanha","msg",
    "ok","oks","vlw","valeu","tipo","pqp","qq","q","pra","p","p/","c/","s/","ta","tá","eh","ehh","mt","mto","bj","bjs",
}
NUMERIC_OK = re.compile(r"^\d+([.,]\d+)?(cm|mm|g|kg|x|x\d+)?$", flags=re.IGNORECASE)
CEP_ONLY_RX = re.compile(r"^\d{5}-?\d{3}$")

def tokenize(text: str):
    t = str(text).lower().strip()
    t = re.sub(r"[^0-9a-záéíóúàâêôãõç\-\s]", " ", t, flags=re.IGNORECASE)
    t = re.sub(r"\s+", " ", t)
    return t.split()

def top_ngrams(lines, n=2, k=100):
    counts = Counter()
    for s in lines:
        toks = tokenize(s)
        if len(toks) < n: continue
        for i in range(len(toks)-n+1):
            counts[tuple(toks[i:i+n])] += 1
    return counts.most_common(k)

def build_lexicon(df: pd.DataFrame, text_col: str, max_vocab=8000):
    counts = Counter()
    for s in df[text_col].astype(str):
        for tok in tokenize(s):
            counts[tok] += 1
    vocab = [w for w,_ in counts.most_common(max_vocab)]
    vocab = list(dict.fromkeys(vocab + list(ABBREV_WHITELIST)))
    return set(vocab), counts

def oov_rate(text: str, vocab: set) -> float:
    toks = tokenize(text)
    if not toks: return 0.0
    oov = 0
    for t in toks:
        if t in vocab: continue
        if t in ABBREV_WHITELIST: continue
        if NUMERIC_OK.match(t) or CEP_ONLY_RX.match(t): continue
        oov += 1
    return oov / max(1, len(toks))

lines_all = df0["texto"].astype(str).tolist()
vocab, freq = build_lexicon(df0, "texto", max_vocab=8000)
bigrams  = top_ngrams(lines_all, n=2, k=50)
trigrams = top_ngrams(lines_all, n=3, k=50)

style_profile = {
    "vocab_size": len(vocab),
    "top_tokens": [w for w,_ in freq.most_common(100)],
    "top_bigrams": [" ".join(ng) for ng,_ in bigrams],
    "top_trigrams": [" ".join(ng) for ng,_ in trigrams],
    "oov_max_rate": OOV_MAX_RATE,
    "abbrev_whitelist": sorted(list(ABBREV_WHITELIST)),
}
(OUT_DIR / "style_profile.json").write_text(json.dumps(style_profile, ensure_ascii=False, indent=2), encoding="utf-8")

hr("STYLE PROFILE")
print("Vocabulário (aprox):", style_profile["vocab_size"])
print("Top tokens:", style_profile["top_tokens"][:20])
print("OOV_MAX_RATE:", style_profile["oov_max_rate"])
print("Ex. bigrams:", style_profile["top_bigrams"][:8])



STYLE PROFILE 
Vocabulário (aprox): 1902
Top tokens: ['a', 'o', 'e', 'que', 'tem', 'no', 'de', 'num', 'com', 'para', 'obrigada', 'um', 'tamanho', 'da', 'verde', 'preto', 'por', 'em', 'na', 'quero']
OOV_MAX_RATE: 0.06
Ex. bigrams: ['tudo bem', 'a c', 'bom dia', 'na o', 'a o', 'no site', 'num num', 'verde no']


In [183]:
# CÉLULA 3.9 — Sanity: garante df0['intent']

import unicodedata, re
def strip_accents(s):
    return "".join(ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch))
def normalize_intent(s: str) -> str:
    s = strip_accents(str(s or "").lower())
    s = re.sub(r"\s+", "_", s).strip("_")
    return s

if "intent" not in df0.columns or df0["intent"].isna().all():
    df0["intent"] = df0["texto"].map(intent_by_rules)
    df0["intent_source"] = "heuristic_auto"

df0["intent"] = df0["intent"].map(normalize_intent)
print("Sanity check aplicado. Colunas:", df0.columns.tolist())
print(df0["intent"].value_counts().head(10))


Sanity check aplicado. Colunas: ['id', 'ts', 'texto', 'intent', 'origem', 'arquivo_origem', 'is_ruido', 'fonte', 'split', 'is_synth_test', 'intent_source']
intent
outros                          279
saudacao                        180
buscar_produto_por_categoria    151
despedida                       118
duvida_tamanho                   88
disponibilidade                  83
buscar_produto_por_nome          39
onde_comprar                     28
pedido_sugestao                  26
troca_devolucao                  17
Name: count, dtype: int64


In [184]:
# CÉLULA 4 — Distribuição atual + duplicadas

TEXT_COL = "texto"; INT_COL = "intent"

LIXO_RX = re.compile(r"^(\.\.|kk+|rs+|👍|👏|👏🏻|😂|😅|😍|amei!?|ameiii!?|ok!?|blz)$", flags=re.IGNORECASE)
df0 = df0[df0[TEXT_COL].astype(str).str.len() >= 2].copy()
df0 = df0[~df0[TEXT_COL].str.fullmatch(LIXO_RX, na=False)].copy()

n_before = len(df0)
df0 = df0.drop_duplicates(subset=[TEXT_COL, INT_COL]).reset_index(drop=True)
dup_exact = n_before - len(df0)

dist = df0.groupby(INT_COL).size().sort_values(ascending=False)
dist_df = dist.rename("count").reset_index().rename(columns={INT_COL:"intent"})
(OUT_DIR / "class_distribution_before.csv").write_text(dist_df.to_csv(index=False), encoding="utf-8")

hr("DISTRIBUIÇÃO ATUAL")
print(dist_df)
print("\nDuplicatas exatas removidas:", dup_exact)
print("Total amostras (limpas):", len(df0))



DISTRIBUIÇÃO ATUAL 
                          intent  count
0                         outros    278
1                       saudacao    180
2   buscar_produto_por_categoria    151
3                      despedida    118
4                 duvida_tamanho     88
5                disponibilidade     83
6        buscar_produto_por_nome     39
7                   onde_comprar     28
8                pedido_sugestao     26
9                troca_devolucao     17
10             tamanho_modelagem     16
11              falar_com_humano     15
12                   frete_prazo     14
13              formas_pagamento     13
14         styling_sugestao_look     11
15              erros_plataforma      6
16                 agradecimento      4
17            materiais_cuidados      4
18                 status_pedido      2

Duplicatas exatas removidas: 0
Total amostras (limpas): 1093


In [185]:
# CÉLULA 5 — (opcional) Alvos por classe para data augmentation "geral"

CONV_INTENTS = {"saudacao","agradecimento","despedida"}
SUPPORT_INTENTS = {"onde_comprar","erros_plataforma","falar_com_humano","cupom_primeira","cliente_loja_b2b","pos_venda"}
OUTROS_NAME = "outros"

CORE_TARGET = 350
SUPPORT_TARGET = 220
CONVERS_TARGET = 180
OUTROS_TARGET = 400

all_intents = set(df0["intent"].unique()) | {normalize_intent(x) for x in yaml_intents}
targets = {}
for it in sorted(all_intents):
    if it == OUTROS_NAME: targets[it] = OUTROS_TARGET
    elif it in CONV_INTENTS: targets[it] = CONVERS_TARGET
    elif it in SUPPORT_INTENTS: targets[it] = SUPPORT_TARGET
    else: targets[it] = CORE_TARGET

targets_df = pd.DataFrame({"intent": list(targets.keys()), "target": list(targets.values())}).sort_values("intent").reset_index(drop=True)
(OUT_DIR / "class_targets.csv").write_text(targets_df.to_csv(index=False), encoding="utf-8")

hr("ALVOS POR CLASSE")
print(targets_df.head(50))



ALVOS POR CLASSE 
                          intent  target
0                  agradecimento     180
1                            any     350
2   buscar_produto_por_categoria     350
3        buscar_produto_por_nome     350
4               cliente_loja_b2b     220
5                      despedida     180
6                disponibilidade     350
7                 duvida_tamanho     350
8               erros_plataforma     220
9               falar_com_humano     220
10              formas_pagamento     350
11                   frete_prazo     350
12            materiais_cuidados     350
13                  onde_comprar     220
14                        outros     400
15               pedido_sugestao     350
16                     pos_venda     220
17                      saudacao     180
18                 status_pedido     350
19         styling_sugestao_look     350
20             tamanho_modelagem     350
21               troca_devolucao     350


In [186]:
# CÉLULA 6 — Slots & Templates por intenção (com trava de estilo)

# Catálogo opcional (se existir em outro notebook, aqui mantemos None)
if "catalog" not in globals():
    catalog = None

# Slots base
TAMANHOS    = ["PP","P","M","G","GG","36","38","40","42","44","46"]
MEDIDAS_B   = ["92cm","96cm","100cm","104cm","110cm","116cm"]
MEDIDAS_Q   = ["100cm","106cm","112cm","118cm","124cm","130cm"]
CEPs        = ["01255-022","01455-022","04567-010","04012-000","05010-020","06030-050"]
OCASIOES    = ["trabalho","casamento de dia","formatura","jantar","aniversário","happy hour"]
ESTILOS     = ["clássico","minimalista","romântico","street","casual","moderno"]
CORES       = ["preto","off-white","azul marinho","vermelho","bege","verde","caramelo"]
PROD_NOMES  = ["vestido midi", "calça alfaiataria", "blazer linho", "saia plissada", "body canelado", "camisa de viscose"]
CATEGORIAS  = ["vestido","saia","blazer","calça","body","camisa","jaqueta","shorts"]

ABBREV_WH = set(style_profile.get("abbrev_whitelist", [])) or ABBREV_WHITELIST
NUMERIC_OK_RX = re.compile(r"^\d+([.,]\d+)?(cm|mm|g|kg|x|x\d+)?$", flags=re.IGNORECASE)
CEP_RX = re.compile(r"^\d{5}-?\d{3}$")

def oov_rate_gate(text: str, vocab_set: set) -> float:
    toks = re.sub(r"[^0-9a-záéíóúàâêôãõç\-\s]", " ", str(text).lower()).split()
    if not toks: return 0.0
    oov = 0
    for t in toks:
        if t in vocab_set or t in ABBREV_WH or NUMERIC_OK_RX.match(t) or CEP_RX.match(t):
            continue
        oov += 1
    return oov / max(1, len(toks))

VOCAB   = set(style_profile.get("top_tokens", []))
LEXICON = set(vocab) | VOCAB

def style_ok(text: str) -> bool:
    return oov_rate_gate(text, LEXICON) <= style_profile.get("oov_max_rate", OOV_MAX_RATE)

def apply_style(text: str, tag: str) -> str:
    if tag == "emoji":
        return (text + " 😊") if "😊" in " ".join(style_profile.get("top_tokens", [])) else (text + " 🙂")
    if tag == "formal":
        return text.replace("vc", "você").replace("qto", "quanto").replace("pq", "porque")
    return text

def inject_typos(text: str, prob=0.06):
    toks = text.split()
    out = []
    for w in toks:
        if np.random.rand() > prob or len(w) < 5:
            out.append(w); continue
        i = np.random.randint(1, len(w)-1)
        out.append(w[:i] + w[i]*2 + w[i+1:])
    return " ".join(out)

TEMPLATES = {
    "saudacao": [
        "oi bia!", "oii tudo bem?", "olá, tudo certo por aí?",
        "bom dia! tem novidades?", "boa tarde, tudo bem com vc?",
    ],
    "agradecimento": [
        "obg!!", "valeu demais", "muito obrigada, ajudou", "perfeito, obrigada!",
    ],
    "despedida": [
        "brigadão, até!", "fechou, obrigada, até mais", "tchau, bom dia!",
    ],
    "duvida_tamanho": [
        "uso {tam} na parte de cima; o {produto} serviria?",
        "tenho {busto} de busto e {quadril} de quadril, qual tamanho do {produto} vc indica?",
        "veste pequeno ou grande? queria o {produto} em {tam}",
    ],
    "tamanho_modelagem": [
        "essa modelagem é mais enxuta? o {produto} veste grande?",
        "tem tabela de medidas do {produto}?",
        "sou {tam}, às vezes pego {tam2}; o {produto} segue a tabela certinha?",
    ],
    "pedido_sugestao": [
        "me indica um {categoria} {cor} tamanho {tam}?",
        "quero um look {estilo} para {ocasiao}, uso {tam}",
        "qual {categoria} combina com {produto} em {cor}?",
    ],
    "styling_sugestao_look": [
        "que {categoria} posso usar com {produto}? quero algo {estilo}",
        "dica de look para {ocasiao} com {produto} {cor}",
    ],
    "buscar_produto_por_categoria": [
        "tem {categoria} {cor} tamanho {tam}?",
        "quero {categoria} {cor} {tam}",
    ],
    "buscar_produto_por_nome": [
        "tem {produto} {cor} {tam}?",
        "procuro {produto} em {cor}",
    ],
    "disponibilidade": [
        "ainda tem {produto} em {tam}?",
        "tem {categoria} {cor} no {tam}?",
    ],
    "preco_pagamento": [
        "quanto tá o {produto} {cor}?",
        "qual o preço do {produto} em {tam}?",
    ],
    "formas_pagamento": [
        "posso pagar no pix/boleto? parcela?",
        "faz em 3x sem juros?",
    ],
    "frete_prazo": [
        "valor do frete pro {cep}?",
        "prazo pro meu cep {cep}, chega até {quando}?",
    ],
    "prazo_entrega": [
        "chega quando no {cep}?",
        "consigo receber até {quando}? meu cep é {cep}",
    ],
    "status_pedido": [
        "meu pedido {pedido} não atualiza, pode ver?",
        "cadê o rastreamento do {pedido}?",
    ],
    "troca_devolucao": [
        "não serviu, consigo trocar?",
        "veio com defeito, como faço a devolução?",
    ],
    "materiais_cuidados": [
        "é {material}? encolhe? como lavar?",
        "posso passar {material}? precisa cuidado especial?",
    ],
    "onde_comprar": [
        "tem link? como compro?",
        "tem loja física/endereço?",
    ],
    "erros_plataforma": [
        "checkout travando aqui",
        "pagamento recusado toda hora",
        "carrinho não abre no celular",
    ],
    "falar_com_humano": [
        "consigo falar com atendente?",
        "tem telefone/whatsapp de alguém?",
    ],
    "cupom_primeira": [
        "tem desconto na primeira compra?",
        "cupom pra novo cliente?",
    ],
    "cliente_loja_b2b": [
        "vocês vendem para lojista/atacado?",
        "posso comprar com cnpj? tem pedido mínimo?",
    ],
    "pos_venda": [
        "chegou faltando peça",
        "rasgou com pouco uso, tem garantia?",
    ],
}

def slot_val(name):
    if name == "tam": return random.choice(TAMANHOS)
    if name == "tam2": return random.choice(TAMANHOS)
    if name == "busto": return random.choice(MEDIDAS_B)
    if name == "quadril": return random.choice(MEDIDAS_Q)
    if name == "produto": return random.choice(PROD_NOMES)
    if name == "categoria": return random.choice(CATEGORIAS)
    if name == "cor": return random.choice(CORES)
    if name == "cep": return random.choice(CEPs)
    if name == "ocasiao": return random.choice(OCASIOES)
    if name == "estilo": return random.choice(ESTILOS)
    if name == "quando": return random.choice(["amanhã","sexta","3 dias úteis","essa semana"])
    if name == "pedido": return "#" + str(random.randint(1000,9999))
    if name == "material": return random.choice(["linho","viscose","algodão","malha canelada"])
    return "?"

def render_template(tpl: str):
    out = tpl
    for m in re.findall(r"{(.*?)}", tpl):
        out = out.replace("{%s}" % m, slot_val(m))
    return out

def synth_for_intent(intent: str, n: int, style_tag="informal"):
    """
    Gera **até n** exemplos novos para a intent.
    Usa while com tentativas extras para superar filtros/dedup.
    """
    rows = []
    tpls = TEMPLATES.get(intent, [])
    base_pool = df0[df0["intent"]==intent]["texto"].astype(str).tolist()
    attempts = 0
    target = n
    # limite duro pra não travar
    max_attempts = max(200, 5*n)

    while len(rows) < target and attempts < max_attempts:
        attempts += 1
        if tpls and (np.random.rand() < 0.85 or not base_pool):
            text = render_template(random.choice(tpls))
        elif base_pool:
            text = inject_typos(random.choice(base_pool), prob=0.08)
        else:
            text = f"exemplo de {intent}"

        # variações pequenas
        if np.random.rand() < 0.25:
            text = text.replace("  ", " ").strip().rstrip(".") + random.choice(["", "!", " :)"])

        text = apply_style(text, style_tag)
        text = text.strip()

        if style_ok(text):
            rows.append({
                "texto": text,
                "intent": intent,
                "source": "synthetic_min",
                "style_tag": style_tag,
                "slots_json": "",
                "quality_flags": "",
            })
    return pd.DataFrame(rows, columns=["texto","intent","source","style_tag","slots_json","quality_flags"])


In [187]:
# === CÉLULA 6B — Atingir metas por intent e sobrescrever NB00 (gera novas linhas) ===
# Pré-requisitos: df0 já carregado e normalizado com colunas ["texto","intent"] (das células 1–4/6)
# Objetivo: garantir totais diferenciados por intent (>=50 e mais volume nas intents core),
#           sem dedupe aproximado (apenas EXATO), e salvar no NB00 oficial.

from pathlib import Path
import re, random, unicodedata
import pandas as pd
import numpy as np
from datetime import datetime

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ---------------------------
# 0) utilitários rápidos
# ---------------------------
def _norm_intent(s: str) -> str:
    s = re.sub(r"\s+", "_", "".join(ch for ch in unicodedata.normalize("NFKD", str(s).lower())
                                    if not unicodedata.combining(ch))).strip("_")
    return s

def _ensure_cols(df, cols):
    df = df.copy()
    for c in cols:
        if c not in df.columns:
            df[c] = ""
    return df[cols].copy()

# ---------------------------
# 1) metas diferenciadas
#    (mantém intents altas como estão; não cresce "outros")
# ---------------------------
# mapa de metas alvo (números variados e com foco nas intents de negócio)
PER_INTENT_TARGET = {
    # núcleo de produto/tamanho/estoque (mais exemplos)
    "duvida_tamanho": 120,
    "disponibilidade": 100,
    "buscar_produto_por_categoria": 170,
    "buscar_produto_por_nome": 90,

    # descoberta/compra
    "pedido_sugestao": 84,
    "onde_comprar": 72,
    "styling_sugestao_look": 60,

    # pós-checkout e dúvidas comuns
    "status_pedido": 66,
    "troca_devolucao": 60,
    "tamanho_modelagem": 60,
    "formas_pagamento": 62,
    "frete_prazo": 61,
    "erros_plataforma": 58,
    "falar_com_humano": 58,
    "materiais_cuidados": 55,
    "agradecimento": 55,

    # conversacionais e “outros”: não aumentar
    "saudacao": None,         # mantém como está (já é alta)
    "despedida": None,        # mantém como está
    "outros": None,           # NÃO aumentar
}

# qualquer intent não listada explicitamente: alvo mínimo 50
DEFAULT_MIN = 50

INTENTS_EXCLUIR = {"cliente_loja_b2b", "pos_venda"}

# ---------------------------
# 2) normaliza base df0 e aplica exclusões
# ---------------------------
df0 = df0.copy()
df0["intent"] = df0["intent"].map(_norm_intent)
before = len(df0)
df0 = df0[~df0["intent"].isin({ _norm_intent(x) for x in INTENTS_EXCLUIR })].copy()
print(f"[6B] removidas intents desativadas: {before-len(df0)} linhas")

# ---------------------------
# 3) templates e slots (reaproveita se já existir; senão cria)
# ---------------------------
if "TEMPLATES" not in globals():
    TEMPLATES = {}

# adiciona (ou completa) templates seguros por intent
TEMPLATES.setdefault("duvida_tamanho", [
    "tenho {busto} de busto e {quadril} de quadril; qual {tam} você indica?",
    "uso {tam} e queria o {produto}; veste grande ou pequeno?",
    "fico entre {tam} e {tam2}; o {produto} tem modelagem mais justa?"
])
TEMPLATES.setdefault("tamanho_modelagem", [
    "essa modelagem é mais enxuta? {produto} costuma vestir grande?",
    "tem tabela de medidas do {produto}?"
])
TEMPLATES.setdefault("disponibilidade", [
    "ainda tem {produto} no {tam}?",
    "tem {categoria} {cor} tamanho {tam}?"
])
TEMPLATES.setdefault("buscar_produto_por_categoria", [
    "tem {categoria} {cor} {tam}?",
    "quero {categoria} {cor} no {tam}"
])
TEMPLATES.setdefault("buscar_produto_por_nome", [
    "tem {produto} {cor} {tam}?",
    "procuro {produto} em {cor}"
])
TEMPLATES.setdefault("pedido_sugestao", [
    "qual {categoria} combina com {produto} {cor}?",
    "me indica um {categoria} {cor} no {tam}?"
])
TEMPLATES.setdefault("onde_comprar", [
    "tem link? como compro?",
    "tem loja física/endereço?"
])
TEMPLATES.setdefault("formas_pagamento", [
    "posso pagar no pix/boleto? parcela?",
    "faz em 3x sem juros?"
])
TEMPLATES.setdefault("frete_prazo", [
    "valor do frete pro {cep}?",
    "prazo pro meu cep {cep}, chega até {quando}?"
])
TEMPLATES.setdefault("status_pedido", [
    "meu pedido {pedido} não atualiza, pode ver?",
    "cadê o rastreio do {pedido}?"
])
TEMPLATES.setdefault("erros_plataforma", [
    "checkout travando aqui",
    "pagamento recusado toda hora",
    "carrinho não abre no celular"
])
TEMPLATES.setdefault("troca_devolucao", [
    "não serviu, consigo trocar?",
    "veio com defeito, como faço a devolução?"
])
TEMPLATES.setdefault("falar_com_humano", [
    "consigo falar com atendente?",
    "tem telefone/whatsapp de alguém?"
])
TEMPLATES.setdefault("materiais_cuidados", [
    "é {material}? encolhe? como lavar?",
    "posso passar {material}? precisa cuidado especial?"
])
TEMPLATES.setdefault("agradecimento", [
    "obg!!", "valeu, obrigada", "perfeito, obrigada!"
])

# slots
PROD_NOMES  = ["vestido midi","calça alfaiataria","blazer linho","saia plissada","body canelado","camisa de viscose"]
CATEGORIAS  = ["vestido","saia","blazer","calça","body","camisa","jaqueta","shorts"]
CORES       = ["preto","off-white","azul marinho","vermelho","bege","verde","caramelo"]
TAMANHOS    = ["PP","P","M","G","GG","36","38","40","42","44","46"]
MEDIDAS_B   = ["92cm","96cm","100cm","104cm","110cm","116cm"]
MEDIDAS_Q   = ["100cm","106cm","112cm","118cm","124cm","130cm"]
CEPs        = ["01255-022","01455-022","04567-010","04012-000","05010-020","06030-050"]
OCASIOES    = ["trabalho","casamento de dia","formatura","jantar","aniversário","happy hour"]
ESTILOS     = ["clássico","minimalista","romântico","street","casual","moderno"]

def _slot(name):
    if name == "tam": return random.choice(TAMANHOS)
    if name == "tam2": return random.choice(TAMANHOS)
    if name == "busto": return random.choice(MEDIDAS_B)
    if name == "quadril": return random.choice(MEDIDAS_Q)
    if name == "produto": return random.choice(PROD_NOMES)
    if name == "categoria": return random.choice(CATEGORIAS)
    if name == "cor": return random.choice(CORES)
    if name == "cep": return random.choice(CEPs)
    if name == "ocasiao": return random.choice(OCASIOES)
    if name == "estilo": return random.choice(ESTILOS)
    if name == "quando": return random.choice(["amanhã","sexta","3 dias úteis","essa semana"])
    if name == "pedido": return "#" + str(random.randint(1000,9999))
    if name == "material": return random.choice(["linho","viscose","algodão","malha canelada"])
    return "?"

def _render(tpl: str) -> str:
    out = tpl
    for m in re.findall(r"{(.*?)}", tpl):
        out = out.replace("{%s}" % m, _slot(m))
    # variações levinhas
    if random.random() < 0.25 and len(out) > 12:
        i = random.randrange(1, len(out)-1)
        out = out[:i] + out[i] + out[i:]
    out = re.sub(r"\s+", " ", out).strip()
    return out

def synth_for_intent_strict(intent: str, n: int) -> pd.DataFrame:
    """Gera N novas frases (sem filtrar por OOV/estilo), evita somente duplicata EXATA com df0."""
    tpls = TEMPLATES.get(intent, [])
    base_pool = df0.loc[df0["intent"]==intent, "texto"].astype(str).tolist()
    already = set((df0["texto"].astype(str) + "||" + df0["intent"].astype(str)).tolist())

    rows = []
    for _ in range(n*3):  # over-generate um pouco pra garantir variedade
        if len(rows) >= n: break
        if tpls:
            text = _render(random.choice(tpls))
        elif base_pool:
            text = random.choice(base_pool)
            # mínima mutação
            text = re.sub(r"\bvc\b", random.choice(["vc","você"]), text)
            if random.random()<0.2 and len(text)>10:
                i = random.randrange(1, len(text)-1)
                text = text[:i] + text[i] + text[i:]
            text = re.sub(r"\s+", " ", text).strip()
        else:
            text = f"exemplo de {intent} {random.randint(1000,9999)}"
        key = f"{text}||{intent}"
        if key not in already and len(text) >= 3:
            rows.append({"texto": text, "intent": intent})
            already.add(key)
    return pd.DataFrame(rows, columns=["texto","intent"])

# ---------------------------
# 4) calcula necessidades e gera
# ---------------------------
cur_counts = df0["intent"].value_counts().to_dict()
need_plan = []
for it in sorted(cur_counts.keys()):
    if it in INTENTS_EXCLUIR: 
        continue
    # alvo:
    if it in ("outros",):  # nunca aumentar "outros"
        target = cur_counts[it]
    else:
        t = PER_INTENT_TARGET.get(it, None)
        target = cur_counts[it] if t is None else max(t, cur_counts[it])
        if t is None:
            target = max(DEFAULT_MIN, cur_counts[it])  # intents desconhecidas => mínimo 50
    need = max(0, target - cur_counts[it])
    need_plan.append((it, cur_counts[it], target, need))

need_df = pd.DataFrame(need_plan, columns=["intent","current","target","need"]).sort_values(["need","intent"], ascending=[False,True]).reset_index(drop=True)
print("\n[6B] PLANO DE GERAÇÃO (antes do synth):")
display(need_df.head(30))
total_need = int(need_df["need"].sum())
print(f"[6B] total a gerar: {total_need}")

parts = []
for it, cur, tgt, need in need_plan:
    if need > 0:
        parts.append(synth_for_intent_strict(it, int(need)))

df_add = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=["texto","intent"])
print("[6B] geradas novas linhas:", len(df_add))

# ---------------------------
# 5) concatena + dedupe EXATO (apenas)
# ---------------------------
df_new = pd.concat([_ensure_cols(df0, ["texto","intent"]), _ensure_cols(df_add, ["texto","intent"])],
                   ignore_index=True)
before = len(df_new)
df_new = df_new.drop_duplicates(subset=["texto","intent"]).reset_index(drop=True)
print(f"[6B] dedupe exato: {before} → {len(df_new)}")

# ---------------------------
# 6) salva no NB00 oficial (apenas texto,intent)
# ---------------------------
NB_DATA_DIR = ROOT / "code" / "notebooks" / "dataset"
NB_DATA_DIR.mkdir(parents=True, exist_ok=True)
NB_DATASET_PATH = NB_DATA_DIR / "dataset_unificado.csv"

df_save = df_new[["texto","intent"]].copy()
df_save.to_csv(NB_DATASET_PATH, index=False, encoding="utf-8")
print(f"[6B] [OK] escrito: {NB_DATASET_PATH} → {len(df_save)} linhas")

# atualiza df0 em memória
df0 = df_save.copy()

# relatório final por classe
final_counts = df0["intent"].value_counts().sort_values(ascending=False)
print("\n[6B] distribuição final:")
display(final_counts.to_frame("count"))
faltando = []
for it, cur, tgt, need in need_plan:
    if cur < tgt:
        # recalc do final
        cur2 = int(final_counts.get(it, 0))
        if cur2 < tgt:
            faltando.append((it, cur2, tgt, tgt-cur2))
if faltando:
    print("⚠️ ainda abaixo do alvo em:", faltando)
else:
    print("✓ metas por intent atingidas (ou preservadas onde já eram maiores).")


[6B] removidas intents desativadas: 0 linhas

[6B] PLANO DE GERAÇÃO (antes do synth):


,intent,current,target,need
0,status_pedido,2,66,64
1,pedido_sugestao,26,84,58
2,erros_plataforma,6,58,52
3,agradecimento,4,55,51
4,buscar_produto_por_nome,39,90,51
5,materiais_cuidados,4,55,51
6,formas_pagamento,13,62,49
7,styling_sugestao_look,11,60,49
8,frete_prazo,14,61,47
9,onde_comprar,28,72,44


[6B] total a gerar: 714
[6B] geradas novas linhas: 555
[6B] dedupe exato: 1648 → 1648
[6B] [OK] escrito: c:\Users\win\Documents\GitHub\2025-2A-T07-CC11-G04\code\notebooks\dataset\dataset_unificado.csv → 1648 linhas

[6B] distribuição final:


,count
intent,
outros,278
saudacao,180
buscar_produto_por_categoria,170
duvida_tamanho,120
despedida,118
disponibilidade,100
buscar_produto_por_nome,90
pedido_sugestao,84
status_pedido,66


⚠️ ainda abaixo do alvo em: [('agradecimento', 17, 55, 38), ('erros_plataforma', 36, 58, 22), ('falar_com_humano', 39, 58, 19), ('formas_pagamento', 31, 62, 31), ('materiais_cuidados', 47, 55, 8), ('onde_comprar', 51, 72, 21), ('troca_devolucao', 40, 60, 20)]


In [188]:
# CÉLULA 7 — (opcional) Aumento geral conforme plano (pode pular)
plan_df = pd.DataFrame(columns=["intent","add_needed"])
df_synth = pd.DataFrame(columns=["texto","intent","source","style_tag","slots_json","quality_flags"])
print("CÉLULA 7: sem aumento geral (usaremos a CÉLULA 14 para metas do NB00).")


CÉLULA 7: sem aumento geral (usaremos a CÉLULA 14 para metas do NB00).


In [189]:
# CÉLULA 8 — Back-translation (off por padrão)

df_bt = pd.DataFrame(columns=["texto","intent","source","style_tag","slots_json","quality_flags"])
print("Back-translation desativado.")


Back-translation desativado.


In [190]:
# CÉLULA 9 — Active Learning (off por padrão)

df_pseudo = pd.DataFrame(columns=["texto","intent","source","style_tag","slots_json","quality_flags"])
print("Active Learning desativado.")


Active Learning desativado.


In [191]:
# CÉLULA 10 — União básica (mantém original para relatórios posteriores)

EXPECTED_COLS = ["texto","intent","source","style_tag","slots_json","quality_flags"]

def ensure_cols(df, cols):
    df = df.copy()
    for c in cols:
        if c not in df.columns:
            df[c] = ""
    return df[cols].copy()

df_orig = df0.copy()
df_orig = ensure_cols(df_orig, ["texto","intent"])
df_orig["source"] = "original"; df_orig["style_tag"] = "orig"; df_orig["slots_json"] = ""; df_orig["quality_flags"] = ""

df_all = df_orig.copy()  # neste fluxo, ainda sem juntar synth/BT/pseudo (a CÉLULA 14 que vai gerar e escrever NB00)
print("df_all (provisório):", df_all.shape)


df_all (provisório): (1648, 6)


In [192]:
# CÉLULA 11 — Relatórios rápidos (pré-ajuste mínimo)

def mean_oov(series):
    vals = [oov_rate(str(s), LEXICON) for s in series.astype(str)]
    return float(np.mean(vals)) if vals else 0.0

rep = []
for src, sub in df_all.groupby("source"):
    rep.append({"source": src, "n": len(sub), "oov_mean": round(mean_oov(sub["texto"]), 4)})
rep_df = pd.DataFrame(rep).sort_values("n", ascending=False)

hr("OOV por source (quanto menor, melhor)")
print(rep_df)



OOV por source (quanto menor, melhor) 
     source     n  oov_mean
0  original  1648     0.119


In [193]:
# CÉLULA 12 — (opcional) Promover dataset canônico como oficial (não usado no NB00)
print("PROMOTE_CANONICAL_AS_OFFICIAL=False — nada será promovido aqui.")


PROMOTE_CANONICAL_AS_OFFICIAL=False — nada será promovido aqui.


In [194]:
# CÉLULA 13 — Resumo rápido do que fizemos até aqui

summary = {
    "inputs": {
        "dataset_original": str(DATASET_PATH),
        "fluxos_yaml": str(FLUXOS_PATH) if FLUXOS_PATH else None,
    },
    "flags": {
        "enable_backtranslation": ENABLE_BACKTRANSLATION,
        "enable_active_learning": ENABLE_ACTIVE_LEARNING,
        "promoted_official": PROMOTE_CANONICAL_AS_OFFICIAL,
        "oov_max_rate": OOV_MAX_RATE,
        "test_ratio": TEST_RATIO,
        "dev_ratio": DEV_RATIO,
    }
}
(OUT_DIR / "augmentation_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
hr("RESUMO")
print(json.dumps(summary, ensure_ascii=False, indent=2))



RESUMO 
{
  "inputs": {
    "dataset_original": "c:\\Users\\win\\Documents\\GitHub\\2025-2A-T07-CC11-G04\\code\\notebooks\\dataset\\dataset_unificado.csv",
    "fluxos_yaml": "c:\\Users\\win\\Documents\\GitHub\\2025-2A-T07-CC11-G04\\code\\resources\\fluxos.yaml"
  },
  "flags": {
    "enable_backtranslation": false,
    "enable_active_learning": false,
    "promoted_official": false,
    "oov_max_rate": 0.06,
    "test_ratio": 0.12,
    "dev_ratio": 0.08
  }
}


In [195]:
# CÉLULA 14 — Mínimo por intent (alvo por classe) + SALVAR no NB00 (sobrescreve dataset_unificado.csv)
# >>> ESTA É A CÉLULA QUE "DE FATO" ESCREVE NOVAS LINHAS NO code/notebooks/dataset/dataset_unificado.csv <<<

# === Parâmetros desta célula ===
# Alvos diferenciados por importância (ajuste livre):
# - core (busca/tamanho/disponibilidade) com metas maiores
# - suporte/checkout/logística com metas médias
# - conversa/fechamento medianas
# - 'outros' não é inflado; só coexistente
TARGETS_MIN = {
    # Core de produto
    "buscar_produto_por_categoria": 150,
    "buscar_produto_por_nome": 90,
    "duvida_tamanho": 120,
    "tamanho_modelagem": 80,
    "disponibilidade": 100,
    "pedido_sugestao": 80,
    "styling_sugestao_look": 70,

    # Suporte/checkout/logística
    "onde_comprar": 80,
    "formas_pagamento": 80,
    "frete_prazo": 80,
    "prazo_entrega": 80,     # se existir no seu dataset
    "status_pedido": 60,
    "erros_plataforma": 60,
    "troca_devolucao": 70,
    "materiais_cuidados": 60,
    "falar_com_humano": 60,

    # Conversacionais
    "saudacao": 90,
    "despedida": 70,
    "agradecimento": 60,

    # fallback para qualquer outra intent não listada
    "_default": 50,
}

# Intents para remover definitivamente do dataset
INTENTS_EXCLUIR = {"cliente_loja_b2b", "pos_venda"}

# Caminho oficial do NB00 que será sobrescrito
NB_DATA_DIR = ROOT / "code" / "notebooks" / "dataset"
NB_DATA_DIR.mkdir(parents=True, exist_ok=True)
NB_DATASET_PATH = NB_DATA_DIR / "dataset_unificado.csv"

hr("MÍNIMO/ALVO POR INTENT + REMOÇÃO DE INTENTS DESATIVADAS + ESCRITA NB00")

def _norm_intent(s: str) -> str:
    s = re.sub(r"\s+", "_", "".join(ch for ch in unicodedata.normalize("NFKD", str(s).lower())
                                    if not unicodedata.combining(ch))).strip("_")
    return s

# 1) Normaliza intent e remove desativadas
df0["intent"] = df0["intent"].map(_norm_intent)
antes = len(df0)
df0 = df0[~df0["intent"].isin({ _norm_intent(x) for x in INTENTS_EXCLUIR })].copy()
print(f"• Intents removidas ({INTENTS_EXCLUIR}): {antes - len(df0)} linhas")

# 2) Calcula necessidades por intent com alvos diferenciados
cont = df0.groupby("intent").size().to_dict()
intents_presentes = sorted(cont.keys())

def alvo_para_intent(it: str) -> int:
    return int(TARGETS_MIN.get(it, TARGETS_MIN["_default"]))

need_rows = []
for it in intents_presentes:
    tgt = alvo_para_intent(it)
    falta = max(0, tgt - cont.get(it, 0))
    need_rows.append({"intent": it, "current": cont.get(it, 0), "target": tgt, "need": falta})

need_df = pd.DataFrame(need_rows).sort_values(["need","target","intent"], ascending=[False, False, True]).reset_index(drop=True)

hr("NECESSIDADES POR INTENT")
display(need_df.head(30))
print("Total novas amostras necessárias (teórico):", int(need_df["need"].sum()))

# 3) Gera sintéticas **até** atingir o alvo por intent
synthetic_parts = []
for _, row in need_df.iterrows():
    it = row["intent"]; need = int(row["need"])
    if need <= 0: 
        continue
    # mistura estilos
    informal = int(round(need * 0.55))
    formal   = int(round(need * 0.30))
    emoji    = need - informal - formal
    dfs = []
    if informal > 0: dfs.append(synth_for_intent(it, informal, "informal"))
    if formal   > 0: dfs.append(synth_for_intent(it, formal,   "formal"))
    if emoji    > 0: dfs.append(synth_for_intent(it, emoji,    "emoji"))
    if dfs:
        synthetic_parts.append(pd.concat(dfs, ignore_index=True))

df_add = (pd.concat(synthetic_parts, ignore_index=True)
          if synthetic_parts else
          pd.DataFrame(columns=["texto","intent","source","style_tag","slots_json","quality_flags"]))

print("Geradas (bruto):", len(df_add))

# 4) Concatena com original (apenas texto/intent), depois dedupe exato + aproximado por intent
def _ensure_cols(df, cols):
    df = df.copy()
    for c in cols:
        if c not in df.columns:
            df[c] = ""
    return df[cols].copy()

orig_tb = _ensure_cols(df0, ["texto","intent"]).assign(source="original")
add_tb  = _ensure_cols(df_add, ["texto","intent"])  # já tem source=synthetic_min

df_new = pd.concat([orig_tb, add_tb], ignore_index=True)

def _normalize_for_dupe(s: str) -> str:
    s = s.lower()
    s = "".join(ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch))
    s = re.sub(r"[^0-9a-z\s-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)
    return s

def _jaccard(a: str, b: str) -> float:
    sa, sb = set(re.sub(r"\W+"," ",a).split()), set(re.sub(r"\W+"," ",b).split())
    if not sa or not sb: return 0.0
    return len(sa & sb) / len(sa | sb)

# Exato
df_new = df_new.drop_duplicates(subset=["texto","intent"]).reset_index(drop=True)

# Aproximado dentro de cada intent (um pouco mais permissivo pra garantir crescimento)
keep_parts = []
for it, sub in df_new.groupby("intent", group_keys=False):
    sub = sub.copy()
    sub["_n"] = sub["texto"].map(_normalize_for_dupe)

    buckets = {}
    sel = []
    for i, row in sub.iterrows():
        key = (len(row["_n"])//12, row["_n"][:24])  # bucketing
        dup = False
        for j in buckets.get(key, []):
            if _jaccard(row["_n"], sub.at[j, "_n"]) >= 0.885:  # 0.885 para segurar diversidade sem matar volume
                dup = True; break
        if not dup:
            sel.append(i)
            buckets.setdefault(key, []).append(i)
    keep_parts.append(sub.loc[sel].drop(columns=["_n"]))

df_new = pd.concat(keep_parts, ignore_index=True)

# 5) Checagem: se alguma intent ainda ficou abaixo da meta, faz uma 2ª rodada light
cont2 = df_new.groupby("intent").size().to_dict()
falta2 = []
for it in intents_presentes:
    tgt = alvo_para_intent(it)
    cur = cont2.get(it, 0)
    if cur < tgt:
        falta2.append((it, tgt-cur))
if falta2:
    hr("2ª RODADA DE GERAÇÃO (com reforço)")
    extra_parts = []
    for it, k in falta2:
        if k <= 0: continue
        extra = synth_for_intent(it, int(k*1.3), "informal")  # margem 30% pra compensar novo dedupe
        extra_parts.append(extra)
    if extra_parts:
        extra_add = pd.concat(extra_parts, ignore_index=True)
        df_new2 = pd.concat([df_new, _ensure_cols(extra_add, ["texto","intent"])], ignore_index=True)
        # dedupe leve novamente por intent
        keep2 = []
        for it, sub in df_new2.groupby("intent", group_keys=False):
            sub = sub.copy()
            sub["_n"] = sub["texto"].map(_normalize_for_dupe)
            buckets = {}
            sel = []
            for i, row in sub.iterrows():
                key = (len(row["_n"])//12, row["_n"][:24])
                dup = False
                for j in buckets.get(key, []):
                    if _jaccard(row["_n"], sub.at[j, "_n"]) >= 0.895:
                        dup = True; break
                if not dup:
                    sel.append(i)
                    buckets.setdefault(key, []).append(i)
            keep2.append(sub.loc[sel].drop(columns=["_n"]))
        df_new = pd.concat(keep2, ignore_index=True)

# 6) Embaralha para não deixar sintéticas agrupadas
df_new = df_new.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

# 7) Salva no dataset do NB00 (SOBRESCREVE)
df_save = df_new[["texto","intent"]].copy()
# segurança: remove intents desativadas
df_save = df_save[~df_save["intent"].isin({ _norm_intent(x) for x in INTENTS_EXCLUIR })].copy()
df_save.to_csv(NB_DATASET_PATH, index=False, encoding="utf-8")

hr("SALVO NB00")
print(f"[OK] Escrito: {NB_DATASET_PATH}  → {len(df_save)} linhas")

# 8) Atualiza df0 em memória
df0 = df_save.copy()

# 9) Relatório final por classe
dist_final = df0["intent"].value_counts().sort_values(ascending=False)
hr("DISTRIBUIÇÃO FINAL (APÓS ALVOS)")
display(dist_final.to_frame("count").head(50))

# Mostra quais ficaram abaixo do alvo estipulado (se algum)
faltando = []
for it, cnt in dist_final.items():
    tgt = alvo_para_intent(it)
    if cnt < tgt:
        faltando.append((it, cnt, tgt))
if faltando:
    print("⚠️ Classes ainda abaixo do alvo (ok se deliberado):")
    display(pd.DataFrame(faltando, columns=["intent","count","target"]))
else:
    print("✓ Todas as intents ativas atingiram seus alvos mínimos.")



MÍNIMO/ALVO POR INTENT + REMOÇÃO DE INTENTS DESATIVADAS + ESCRITA NB00 
• Intents removidas ({'pos_venda', 'cliente_loja_b2b'}): 0 linhas

NECESSIDADES POR INTENT 


,intent,current,target,need
0,formas_pagamento,31,80,49
1,agradecimento,17,60,43
2,troca_devolucao,40,70,30
3,onde_comprar,51,80,29
4,erros_plataforma,36,60,24
5,falar_com_humano,39,60,21
6,tamanho_modelagem,60,80,20
7,frete_prazo,61,80,19
8,materiais_cuidados,47,60,13
9,styling_sugestao_look,60,70,10


Total novas amostras necessárias (teórico): 258
Geradas (bruto): 211

2ª RODADA DE GERAÇÃO (com reforço) 

SALVO NB00 
[OK] Escrito: c:\Users\win\Documents\GitHub\2025-2A-T07-CC11-G04\code\notebooks\dataset\dataset_unificado.csv  → 1633 linhas

DISTRIBUIÇÃO FINAL (APÓS ALVOS) 


,count
intent,
outros,278
saudacao,180
buscar_produto_por_categoria,170
duvida_tamanho,119
despedida,118
disponibilidade,100
buscar_produto_por_nome,90
pedido_sugestao,84
status_pedido,66


⚠️ Classes ainda abaixo do alvo (ok se deliberado):


,intent,count,target
0,duvida_tamanho,119,120
1,tamanho_modelagem,61,80
2,frete_prazo,60,80
3,styling_sugestao_look,60,70
4,onde_comprar,49,80
5,materiais_cuidados,40,60
6,troca_devolucao,39,70
7,falar_com_humano,38,60
8,erros_plataforma,35,60
9,formas_pagamento,31,80


In [197]:
# CÉLULA 15 — (opcional) Splits e relatórios pós-NB00 (não reescreve o NB00)

# Vamos só produzir um split rápido para análise local (sem gravar por cima do NB00).
TEST_RATIO = 0.12
DEV_RATIO  = 0.08

df_all_after = df0.copy()
df_all_after["source"] = "mixed"  # original + synthetic, mas NB00 guarda apenas texto/intent

def stratified_split(df: pd.DataFrame, test_ratio: float, dev_ratio: float):
    test_parts, dev_parts, train_parts = [], [], []
    for it, sub in df.groupby("intent"):
        sub = sub.sample(frac=1.0, random_state=SEED)
        n = len(sub)
        n_test = int(round(n * test_ratio))
        n_dev  = int(round(n * dev_ratio))
        test = sub.head(n_test)
        remain = sub.drop(test.index)
        dev  = remain.head(n_dev)
        train= remain.drop(dev.index)
        test_parts.append(test); dev_parts.append(dev); train_parts.append(train)
    return (pd.concat(train_parts, ignore_index=True),
            pd.concat(dev_parts, ignore_index=True),
            pd.concat(test_parts, ignore_index=True))

df_train, df_dev, df_test = stratified_split(df_all_after, TEST_RATIO, DEV_RATIO)

hr("TAMANHOS (pós-NB00, apenas análise)")
print("Train/Dev/Test:", len(df_train), len(df_dev), len(df_test))
print(df_all_after["intent"].value_counts().head(20))



TAMANHOS (pós-NB00, apenas análise) 
Train/Dev/Test: 1307 130 196
intent
outros                          278
saudacao                        180
buscar_produto_por_categoria    170
duvida_tamanho                  119
despedida                       118
disponibilidade                 100
buscar_produto_por_nome          90
pedido_sugestao                  84
status_pedido                    66
tamanho_modelagem                61
frete_prazo                      60
styling_sugestao_look            60
onde_comprar                     49
materiais_cuidados               40
troca_devolucao                  39
falar_com_humano                 38
erros_plataforma                 35
formas_pagamento                 31
agradecimento                    15
Name: count, dtype: int64
